# ResQue: Warm-Start QRC for Multi-Output Weather Forecasting over East Africa
### ResQue | GIC 2026 | Track B: Weather Time-Series Forecasting

[![Launch on qBraid](https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png)](https://account.qbraid.com?gitHubUrl=https://github.com/Armstrong66/resque-qrc)

**This notebook is the Phase 3 entry point. Run cells sequentially.**  
All results written to `outputs/results/`. Full benchmark tables match the write-up.

| Setting | Value |
|---|---|
| Station | NOAA ISD 63450099999 (Addis Ababa Bole, Ethiopia) |
| Horizons | 6h and 24h |
| Primary reservoir | 9-qubit transverse-field Ising chain |
| Encoding ablation | Standard vs. data reuploading (Pérez-Salinas et al. 2020) |
| Warm-start | ESN → QRC via truncated SVD |
| Hardware target | PennyLane lightning.qubit (GPU sim) / QuEra Aquila or IBM Eagle |

**AI disclosure**: Claude (Anthropic) used for code scaffolding — disclosed per GIC rules.

---
## Cell 1 — Environment setup

In [ ]:
# ── Install any missing dependencies ──────────────────────────────────────────
# qBraid Lab has PennyLane, torch, numpy, pandas pre-installed.
# pmdarima and pyarrow may need installing.
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['pmdarima', 'pyarrow', 'pennylane-lightning']:
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
    except ImportError:
        print(f'Installing {pkg}...')
        _install(pkg)

# ── Confirm versions ──────────────────────────────────────────────────────────
import pennylane as qml
import torch
import numpy as np
import pandas as pd

print(f'PennyLane : {qml.__version__}')
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')
print(f'NumPy     : {np.__version__}')

# ── Add project root to path ──────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'\nProject root: {PROJECT_ROOT}')
print('Environment ready.')

---
## Cell 2 — Configuration
All hyperparameters live in `config.py`. Override here for quick experiments.

In [ ]:
from config import *

# ── Override defaults for Phase 3 full run ────────────────────────────────────
# Change these to reproduce specific results from the write-up.

RUN_MODE = 'full'          # 'smoke' (fast, 300 steps) | 'full' (complete dataset)
N_QUBITS_PRIMARY = 9       # Primary result qubit count
J_STAR  = 0.3              # From Phase 2 Hamiltonian sweep
H_STAR  = 0.8              # From Phase 2 Hamiltonian sweep
P_STAR  = 0.0              # From Phase 2 noise sweep (noiseless optimal in sim)
TOPOLOGY = 'chain'         # chain outperformed all_to_all in Phase 2
USE_REUPLOADING = True     # Phase 3 primary encoding — set False for standard ablation

SWEEP_STEPS = 300 if RUN_MODE == 'smoke' else None  # None = full dataset

print(f'Run mode         : {RUN_MODE}')
print(f'Qubits (primary) : {N_QUBITS_PRIMARY}')
print(f'J*, h*           : {J_STAR}, {H_STAR}')
print(f'Data reuploading : {USE_REUPLOADING}')
print(f'Sweep steps      : {"full dataset" if SWEEP_STEPS is None else SWEEP_STEPS}')

---
## Cell 3 — Data download
NOAA ISD global-hourly archive. No API key required.

In [ ]:
from data.downloader import download_all

raw_paths = download_all()
print(f'\n{len(raw_paths)} year files ready.')

---
## Cell 4 — Parse, clean, and inspect
ISD sentinel replacement → Magnus RH → 6h resample → ffill/bfill → dropna

In [ ]:
from data.parser import load_and_merge

df = load_and_merge(raw_paths)

print(f'Timesteps  : {len(df)}')
print(f'Date range : {df.index.min()} → {df.index.max()}')
print(f'\nNaN check (must all be 0%):')
for col in TARGETS:
    pct = df[col].isna().mean() * 100
    flag = '✓' if pct == 0 else f'✗ {pct:.1f}% — DELETE PARQUET AND RERUN'
    print(f'  {col:<20} {flag}')

print(f'\nDescriptive stats (physical units):')
display(df[TARGETS].describe().round(2))

---
## Cell 5 — Preprocessing: shared PCA + windowing

In [ ]:
from preprocessing.pipeline import WeatherPreprocessor

if RUN_MODE == 'smoke':
    df_use = df.iloc[:500]
    print('Smoke mode: using first 500 timesteps')
else:
    df_use = df

prep = WeatherPreprocessor(df_use)
datasets = prep.build_all()
prep.save(datasets)

for h, ds in datasets.items():
    print(ds.summary())

ds6  = datasets[6]
ds24 = datasets[24]
print('\nDatasets ready.')

---
## Cell 6 — Classical baselines
Persistence → ARIMA → ESN → LSTM → GRU  
ESN also provides warm-start weights for QRC readout.

In [ ]:
from baselines.classical import (run_persistence, run_arima,
                                  run_esn, run_rnn)
import time

baseline_results = {}
fitted_esn = None
X_train_esn = None

# Persistence
r = run_persistence(ds6.y_val, ds6.y_test, ds6.X_val, ds6.X_test, window=WINDOW_SIZE)
baseline_results['persistence'] = r
print(f'Persistence  test RMSE (mean): {r.test_rmse.mean():.4f}')

# ARIMA
r = run_arima(ds6.y_train, ds6.y_val, ds6.y_test, TARGETS)
if r: baseline_results['arima'] = r
if r: print(f'ARIMA        test RMSE (mean): {r.test_rmse.mean():.4f}')

# ESN — CRITICAL baseline; also generates warm-start weights
t0 = time.time()
r_esn, fitted_esn = run_esn(ds6.X_train, ds6.y_train,
                              ds6.X_val,   ds6.y_val,
                              ds6.X_test,  ds6.y_test)
baseline_results['esn'] = r_esn
X_train_esn = fitted_esn.get_reservoir_states(ds6.X_train)
print(f'ESN          test RMSE (mean): {r_esn.test_rmse.mean():.4f}  [{time.time()-t0:.1f}s]')

# LSTM
t0 = time.time()
r = run_rnn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val,
            ds6.X_test,  ds6.y_test,  window=WINDOW_SIZE, model_type='lstm')
if r:
    baseline_results['lstm'] = r
    print(f'LSTM         test RMSE (mean): {r.test_rmse.mean():.4f}  [{time.time()-t0:.1f}s]')

# GRU
t0 = time.time()
r = run_rnn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val,
            ds6.X_test,  ds6.y_test,  window=WINDOW_SIZE, model_type='gru')
if r:
    baseline_results['gru'] = r
    print(f'GRU          test RMSE (mean): {r.test_rmse.mean():.4f}  [{time.time()-t0:.1f}s]')

---
## Cell 7 — Encoding ablation: standard vs. data reuploading
**Phase 3 primary experiment.** Compares single-injection vs. re-encoding at every Trotter step.

In [ ]:
from reservoir.quantum_reservoir import encoding_ablation

encoding_results = encoding_ablation(
    X_train=ds6.X_train, y_train=ds6.y_train,
    X_val=ds6.X_val,     y_val=ds6.y_val,
    n_qubits=N_QUBITS_PRIMARY, J=J_STAR, h=H_STAR,
    max_steps=SWEEP_STEPS,
    out_dir=RESULTS
)

print(f"\nEncoding ablation (n={N_QUBITS_PRIMARY} qubits, J={J_STAR}, h={H_STAR}):")
for enc, rmse in encoding_results.items():
    print(f"  {enc:<22} val_rmse = {rmse:.4f}")

improvement = encoding_results.get('standard', 0) - encoding_results.get('data_reuploading', 0)
print(f"\n  Reuploading improvement: {improvement:+.4f} (positive = reuploading wins)")

---
## Cell 8 — QRC training: cold-start and warm-start
Uses the encoding strategy selected above.

In [ ]:
from reservoir.quantum_reservoir import IsingQRC
from readout.ridge_readout import RidgeReadout
import json, pickle

qrc_results = {}

for label, use_warm in [('cold_start_qrc', False), ('warm_start_qrc', True)]:
    print(f'\n--- Training: {label} ---')
    t0 = time.time()
    try:
        qrc = IsingQRC(
            n_qubits=N_QUBITS_PRIMARY, J=J_STAR, h=H_STAR,
            topology=TOPOLOGY, noise_rate=P_STAR,
            use_data_reuploading=USE_REUPLOADING
        )

        H_train = qrc.run_sequence(ds6.X_train, max_steps=SWEEP_STEPS, verbose=True)
        H_val   = qrc.run_sequence(ds6.X_val)
        H_test  = qrc.run_sequence(ds6.X_test)

        n_tr = min(len(H_train), len(ds6.y_train))
        n_vl = min(len(H_val),   len(ds6.y_val))
        n_ts = min(len(H_test),  len(ds6.y_test))

        readout = RidgeReadout(
            target_names=TARGETS,
            warm_start=(use_warm and X_train_esn is not None)
        )
        best = readout.fit(
            H_train[:n_tr], ds6.y_train[:n_tr],
            H_val[:n_vl],   ds6.y_val[:n_vl],
            X_train_esn=X_train_esn[:n_tr] if X_train_esn is not None else None
        )
        readout.save_selection_log(RESULTS)

        pred_test = best.predict(H_test[:n_ts])
        pred_val  = best.predict(H_val[:n_vl])

        # Store for metrics table
        class _R:
            def __init__(self, pv, pt):
                self.y_pred_val  = pv
                self.y_pred_test = pt
        qrc_results[label] = _R(pred_val, pred_test)

        # Save config
        cfg = qrc.get_config()
        cfg.update({'readout_strategy': best.strategy, 'warm_start': use_warm,
                    'val_rmse_mean': float(best.val_rmse_mean),
                    'wall_clock_s': round(time.time()-t0, 1)})
        with open(RESULTS / f'{label}_config.json', 'w') as f:
            json.dump(cfg, f, indent=2)

        print(f'  Done. Strategy={best.strategy} val_rmse={best.val_rmse_mean:.4f} '
              f'[{time.time()-t0:.0f}s]')

    except Exception as e:
        import traceback
        print(f'  FAILED: {e}')
        traceback.print_exc()

---
## Cell 9 — Qubit scaling study
Required by challenge: characterise performance across n = 5 → 20.

In [ ]:
from experiments.sweeps import qubit_scaling_study

df_scaling = qubit_scaling_study(
    ds6.X_train, ds6.y_train,
    ds6.X_val,   ds6.y_val,
    J=J_STAR, h=H_STAR, p=P_STAR,
    qubit_counts=QUBIT_COUNTS   # [5, 7, 9, 12, 16, 20]
)

print('\nQubit scaling results:')
display(df_scaling[['n_qubits', 'feature_dim', 'val_rmse']].to_string(index=False))

---
## Cell 10 — Full benchmark table
All models, both horizons, RMSE + MAE in physical units + VPT.

In [ ]:
from evaluation.metrics import build_results_table

all_results = {**baseline_results, **qrc_results}

print('=== 6-HOUR HORIZON ===')
df_6h = build_results_table(
    results=all_results,
    y_true_val=ds6.y_val,
    y_true_test=ds6.y_test,
    target_names=TARGETS,
    horizon_hours=6,
    out_dir=RESULTS
)
display(df_6h)

# ── Repeat for 24h horizon ────────────────────────────────────────────────────
print('\n=== 24-HOUR HORIZON ===')
# Re-run QRC on ds24 if full run mode
if RUN_MODE == 'full':
    print('(24h QRC run — this will take a while, same as Cell 8 but on ds24)')
    # Add 24h QRC here following same Cell 8 pattern with ds24

df_24h = build_results_table(
    results=baseline_results,  # Add 24h QRC results when available
    y_true_val=ds24.y_val,
    y_true_test=ds24.y_test,
    target_names=TARGETS,
    horizon_hours=24,
    out_dir=RESULTS
)
display(df_24h)

---
## Cell 11 — Noise sweep
Tests whether hardware-induced noise improves generalisation (Antoncich et al. 2026).

In [ ]:
from experiments.sweeps import noise_sweep

p_star, df_noise = noise_sweep(
    ds6.X_train, ds6.y_train,
    ds6.X_val,   ds6.y_val,
    J=J_STAR, h=H_STAR, n_qubits=N_QUBITS_PRIMARY
)

print(f'\nOptimal noise rate p* = {p_star}')
display(df_noise)

noiseless = df_noise[df_noise.noise_rate == 0.0].val_rmse.values[0]
if p_star > 0:
    best_noisy = df_noise[df_noise.noise_rate == p_star].val_rmse.values[0]
    print(f'Noise-assisted: {noiseless:.4f} (p=0) → {best_noisy:.4f} (p={p_star})')
else:
    print(f'Noiseless optimal: {noiseless:.4f}. Hardware test may differ.')

---
## Cell 12 — Shot budget ablation
Confirms RMSE stability from exact simulation through 500→5000 shots.

In [ ]:
from experiments.sweeps import shot_ablation

df_shots = shot_ablation(
    ds6.X_train, ds6.y_train,
    ds6.X_val,   ds6.y_val,
    J=J_STAR, h=H_STAR, p=P_STAR, n_qubits=N_QUBITS_PRIMARY
)

print('Shot ablation results:')
display(df_shots)

---
## Cell 13 — Hardware run (QuEra Aquila or IBM Eagle)
Run this cell only when QPU access is confirmed via qBraid.
Comment out and use IBM fallback if Aquila queue is long.

In [ ]:
# ── HARDWARE SELECTION ────────────────────────────────────────────────────────
# Uncomment ONE of the following backends.

HARDWARE_BACKEND = 'simulation'  # Change to 'aquila' or 'ibm' when QPU ready

if HARDWARE_BACKEND == 'aquila':
    # QuEra Aquila via Bloqade / Amazon Braket
    # Rydberg blockade = native Ising ZZ coupling — no transpilation needed
    print('QuEra Aquila selected. Ensure qBraid Bloqade kernel is active.')
    print('See: github.com/QuEraComputing/QRC-tutorials for Aquila job submission.')
    # TODO (coder): integrate Bloqade job submission here following QuEra QRC tutorials
    # Expected: n=9, n=12 runs at 1000-5000 shots/step

elif HARDWARE_BACKEND == 'ibm':
    # IBM Eagle/Heron via Qiskit Runtime — fallback if Aquila unavailable
    print('IBM backend selected. Ensure qBraid IBM token is configured.')
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
        service = QiskitRuntimeService()  # uses saved token
        backends = service.backends(filters=lambda b: b.num_qubits >= 9
                                    and b.status().operational)
        print(f'Available backends: {[b.name for b in backends[:5]]}')
        # TODO (coder): compile Ising circuit to IBM gate set and submit
        # Use Mitiq ZNE for noise mitigation: pip install mitiq
    except Exception as e:
        print(f'IBM connection failed: {e}')
        print('Check qBraid IBM token configuration.')

else:
    print('Hardware run skipped — using simulation results only.')
    print('QPU access: request via qBraid platform once team account credits confirmed.')

---
## Cell 14 — Summary and output file list
All output files needed for the write-up are listed here.

In [ ]:
from config import RESULTS
import json

output_files = list(RESULTS.glob('*'))
print(f'Output files in {RESULTS}:')
for f in sorted(output_files):
    size_kb = f.stat().st_size // 1024 if f.exists() else 0
    print(f'  {f.name:<45} {size_kb:>6} KB')

# Print key numbers for write-up
print('\n=== KEY NUMBERS FOR WRITE-UP ===')
try:
    cfg = json.load(open(RESULTS / 'warm_start_qrc_config.json'))
    print(f'QRC config: n={cfg["n_qubits"]} J={cfg["J"]} h={cfg["h"]} '
          f'encoding={"data_reuploading" if cfg["use_data_reuploading"] else "standard"}')
    print(f'Circuit depth: {cfg["trotter_steps"]} Trotter steps '
          f'(effective depth ~{cfg["trotter_steps"] * cfg["n_qubits"]})')
    print(f'Wall-clock training time: {cfg.get("wall_clock_s", "N/A")}s')
    print(f'Readout strategy: {cfg["readout_strategy"]}')
    print(f'Val RMSE (mean, normalised): {cfg["val_rmse_mean"]:.4f}')
except FileNotFoundError:
    print('(Run Cell 8 first to generate QRC config)')

print('\nPhase 3 checklist:')
checks = [
    ('results_h6.csv exists',    (RESULTS / 'results_h6.csv').exists()),
    ('results_h24.csv exists',   (RESULTS / 'results_h24.csv').exists()),
    ('qubit_scaling.csv exists', (RESULTS / 'qubit_scaling.csv').exists()),
    ('noise_sweep.csv exists',   (RESULTS / 'noise_sweep.csv').exists()),
    ('shot_ablation.csv exists', (RESULTS / 'shot_ablation.csv').exists()),
    ('encoding_ablation.json',   (RESULTS / 'encoding_ablation.json').exists()),
    ('warm_start_qrc_config',    (RESULTS / 'warm_start_qrc_config.json').exists()),
]
for name, ok in checks:
    print(f'  {"✓" if ok else "✗"} {name}')